# RSNA Knee — ITER-02 fold-0 checks

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-015/016** | `hlly34/rsna-knee-iter02-b` | ITER-02 fold 0: CoAtNet-RMLP-2 @384 (EXP-015, GPU0, 10 epochs, bs 4, val every 2, 9 h cap) and EXP-009 fold 0 re-scored with 64 vs 48 eval windows (EXP-016, GPU1) |

Each GPU runs its own queue of jobs (logs in `/kaggle/working/logs/<exp>.txt`). Training jobs save `models/<exp_name>/fold0.pt`, OOF + gold predictions and a CV json; the summary compares every job with the current best's fold 0 (EXP-009: OOF 0.865, gold 0.876).

| Exp | GPU | Job |
|---|---|---|
| EXP-015 | GPU 0 | train `coatnet_384` fold 0 training.num_epochs=10 training.batch_size=4 training.val_every=2 training.max_time=00:09:00:00 |
| EXP-016 | GPU 1 | re-score `effnet_b0_224/fold0.pt` with k_eval [48, 64] |

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-015/016", "notebook": "hlly34/rsna-knee-iter02-b", "what": "ITER-02 fold 0: CoAtNet-RMLP-2 @384 (EXP-015, GPU0, 10 epochs, bs 4, val every 2, 9 h cap) and EXP-009 fold 0 re-scored with 64 vs 48 eval windows (EXP-016, GPU1)"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
JOBS = [{"exp": "EXP-015", "gpu": 0, "train": ["coatnet_384", 0, ["training.num_epochs=10", "training.batch_size=4", "training.val_every=2", "training.max_time=00:09:00:00"]]}, {"exp": "EXP-016", "gpu": 1, "eval": ["effnet_b0_224/fold0.pt", [48, 64]]}]
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=6):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification", "lightning",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=True)
import torch, timm, pydicom
print("torch", torch.__version__, "| timm", timm.__version__, "| pydicom", pydicom.__version__,
      "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "| CPUs:", os.cpu_count())
DATA = "/kaggle/tmp/data"            # scratch space (not saved); outputs go to /kaggle/working
os.makedirs(f"{DATA}/labels", exist_ok=True)
if not os.path.exists(f"{DATA}/raw"):
    os.symlink(COMP, f"{DATA}/raw")
for name in ["llm_labels_v4_blend.csv", "llm_labels_full.csv", "report_labels_v2.csv"]:
    src = walk_find(lambda d, f: os.path.join(d, name) if name in f else None)
    print(f"{name:28s} <- {src}")
    if src and not os.path.exists(f"{DATA}/labels/{name}"):
        os.symlink(src, f"{DATA}/labels/{name}")
ENV = dict(os.environ, DATA_DIR=DATA, PYTHONPATH=CODE)
os.chdir("/kaggle/working")

def run(*args, env=None, log=None):
    """Run a repo script/module as a subprocess (fresh CUDA context, clean logs)."""
    t = time.time()
    out = open(log, "w") if log else None
    subprocess.run([sys.executable, *args], env=env or ENV, check=True, stdout=out, stderr=subprocess.STDOUT if out else None)
    print(f"[{' '.join(map(str, args[:2]))}] {time.time() - t:.0f}s")

CACHE = walk_find(lambda d, f: d if os.path.basename(d) == "cache_256" and any(x.endswith(".npz") for x in f[:50]) else None)
print("cache:", CACHE, len(os.listdir(CACHE)), "studies")

## Run the job queues (one thread per GPU)

In [ ]:
import threading, psutil, torch
n_gpu = max(1, torch.cuda.device_count())
workers = max(2, os.cpu_count() // n_gpu)
os.makedirs("logs", exist_ok=True)
t0 = time.time(); status = {}

def command(j):
    if "train" in j:
        cfg, fold, extra = j["train"]
        return [sys.executable, "-m", "src.models.train_pl", f"exp={cfg}", f"fold={fold}",
                f"resources.num_workers={workers}", f"paths.cache_dir={CACHE}", *extra]
    ck_rel, ks = j["eval"]
    ck = walk_find(lambda d, f: os.path.join(d, os.path.basename(ck_rel)) if d.endswith(os.path.dirname(ck_rel))
                   and os.path.basename(ck_rel) in f else None)
    return [sys.executable, f"{CODE}/scripts/eval_ckpt.py", "--ckpt", ck, "--k_eval", *map(str, ks),
            "--out", f"/kaggle/working/eval_{j['exp']}.json", f"paths.data_dir={DATA}", f"paths.cache_dir={CACHE}"]

def gpu_worker(gpu):
    for j in [j for j in JOBS if j["gpu"] % n_gpu == gpu]:
        t = time.time()
        with open(f"logs/{j['exp']}.txt", "w") as log:
            rc = subprocess.run(command(j), env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=log,
                                stderr=subprocess.STDOUT).returncode
        status[j["exp"]] = (rc, round((time.time() - t) / 60, 1))
        print(f"{(time.time() - t0) / 60:6.1f} min  {j['exp']} on GPU {gpu} exit {rc} ({status[j['exp']][1]} min)", flush=True)

threads = [threading.Thread(target=gpu_worker, args=(g,)) for g in range(n_gpu)]
[t.start() for t in threads]
while any(t.is_alive() for t in threads):
    time.sleep(600)
    vm = psutil.virtual_memory()
    print(f"{(time.time() - t0) / 60:6.1f} min  RAM {vm.used / 2**30:.1f}/{vm.total / 2**30:.1f} GiB  running", flush=True)
print(status)

## Summary vs EXP-009 fold 0 (OOF 0.865 / gold 0.876)

In [ ]:
import pandas as pd
rows = []
for j in JOBS:
    r = {"exp": j["exp"], "exit": status.get(j["exp"], ("?", 0))[0], "minutes": status.get(j["exp"], ("?", 0))[1]}
    if "train" in j:
        from src.config import get_config
        name = get_config([f"exp={j['train'][0]}", *j["train"][2]]).exp_name
        f = f"models/{name}/fold{j['train'][1]}_metrics.json"
        if os.path.exists(f):
                m = json.load(open(f))
                run(f"{CODE}/scripts/merge_oof.py", "--exp", name, "--oof_dir", "models/oof", "--train", f"{COMP}/train.csv",
                    "--targets", "models/oof/report_targets.csv")
                cv = json.load(open(f"models/oof/{name}_cv.json"))
                r.update(name=name, backbone=m["backbone"], img=m["img_size"], epochs=m["epochs"], oof=cv.get("oof_auc"),
                         gold=cv.get("gold_auc_mean_fold"), lat_men_oof=cv.get("oof_per_label", {}).get("Lateral Meniscus"),
                         lat_men_gold=cv.get("gold_per_label", {}).get("Lateral Meniscus"), topk=m.get("topk_scores"))
    else:
        f = f"/kaggle/working/eval_{j['exp']}.json"
        if os.path.exists(f):
            for k, v in json.load(open(f))["results"].items():
                r[f"oof@k{k}"] = v["oof_auc"]; r[f"gold@k{k}"] = v["gold_auc"]
                r[f"lat_men_oof@k{k}"] = v["oof_per_label"].get("Lateral Meniscus")
    rows.append(r)
summary = pd.DataFrame(rows)
print(summary.to_string())
EXP.update(summary=summary.to_dict("records"), minutes=round((time.time() - t0) / 60, 1))
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1, default=str)